In [4]:
"""
Short version: Essential ranking stability tests (with p-values)
"""

import pandas as pd
import numpy as np
from scipy import stats
from scipy.stats import friedmanchisquare, spearmanr

# Load data
df = pd.read_excel('Axelrod_First_Ranks.xlsx')
rank_cols = [col for col in df.columns if col.startswith('Rank_')]
rankings = df[rank_cols].values  # Shape: (16 strategies, 30 iterations)

print("="*60)
print("RANKING STABILITY TESTS")
print("="*60)
print(f"Data: {rankings.shape[0]} strategies × {rankings.shape[1]} iterations\n")

# 1. KENDALL'S W
n, k = rankings.shape
rank_sums = rankings.sum(axis=1)
mean_rank_sum = rank_sums.mean()
S = np.sum((rank_sums - mean_rank_sum)**2)
W = (12 * S) / (k**2 * (n**3 - n))
chi_square = k * (n - 1) * W
p_value_w = 1 - stats.chi2.cdf(chi_square, n - 1)

print("1. KENDALL'S W (Coefficient of Concordance)")
print(f"   W = {W:.4f}")
print(f"   χ² = {chi_square:.2f}, df = {n-1}, p = {p_value_w}")
print(f"   Interpretation: {'Very Strong' if W > 0.8 else 'Strong' if W > 0.6 else 'Moderate'} agreement\n")

# 2. FRIEDMAN TEST
friedman_stat, friedman_p = friedmanchisquare(*rankings)

print("2. FRIEDMAN TEST")
print(f"   χ² = {friedman_stat:.2f}, df = {n-1}, p = {friedman_p}")

# 3. SPEARMAN CORRELATIONS (with p-values)
correlations = []
p_values = []

for i in range(k):
    for j in range(i+1, k):
        rho, p = spearmanr(rankings[:, i], rankings[:, j])
        correlations.append(rho)
        p_values.append(p)

# Bonferroni correction
bonferroni_alpha = 0.05 / len(correlations)
significant_after_correction = np.sum(np.array(p_values) < bonferroni_alpha)

print("")
print("3. SPEARMAN RANK CORRELATIONS (Pairwise)")
print(f"   Number of pairs: {len(correlations)}")
print(f"   Mean ρ: {np.mean(correlations):.4f}")
print(f"   Range: [{np.min(correlations):.4f}, {np.max(correlations):.4f}]")
print(f"   Mean p-value: {np.mean(p_values):.4e}")
print(f"   Significant at α=0.05: {np.sum(np.array(p_values) < 0.05)}/{len(p_values)}")
print(f"   Bonferroni corrected α: {bonferroni_alpha:.6f}")
print(f"   Significant after correction: {significant_after_correction}/{len(p_values)}")

RANKING STABILITY TESTS
Data: 16 strategies × 30 iterations

1. KENDALL'S W (Coefficient of Concordance)
   W = 0.9711
   χ² = 437.00, df = 15, p = 0.0
   Interpretation: Very Strong agreement

2. FRIEDMAN TEST
   χ² = 436.62, df = 15, p = 1.3593203873036234e-83

3. SPEARMAN RANK CORRELATIONS (Pairwise)
   Number of pairs: 435
   Mean ρ: 0.9693
   Range: [0.8912, 1.0000]
   Mean p-value: 4.4654e-08
   Significant at α=0.05: 435/435
   Bonferroni corrected α: 0.000115
   Significant after correction: 435/435
